# مقایسه Feature Set ها برای پیش‌بینی Phonon Dispersion — دیتاست MAX Phase

**هدف این Notebook:**
1. ساخت دیتاست واحد از 358 ماده MAX Phase (POSCAR + Force Constants + Elastic Constants)
2. آزمایش چند Feature Set اتمی مختلف (که در پروژه‌های قبلی روی Matbench استفاده شدند) — این‌بار روی دیتاست واقعی پروژه
3. آموزش بهترین معماری شناخته‌شده تا الان: **Dual Graph GNN** (Bond Graph + Atom Graph, Attention-based Message Passing)
4. مقایسه دقیق MAE هر Feature Set و انتخاب بهترین

**ریپوی کد اصلی:** [Metisa811/Predict-Phonon-Dispersion-by-GNN-Model](https://github.com/Metisa811/Predict-Phonon-Dispersion-by-GNN-Model)

**منبع معماری Dual Graph:** نسخه‌های `bestbest 9ff018` و `bestbest ce9993` در [Metisa811/kaggle](https://github.com/Metisa811/kaggle)

---
⚠️ **توجه:** مسیرهای Kaggle input (`/kaggle/input/...`) را با توجه به دیتاست‌های واقعی آپلودشده‌ی خودتان تنظیم کنید.


## 1. نصب پکیج‌های لازم

In [ ]:
!pip install -q torch_geometric pymatgen torch-cluster torch-sparse torch-scatter torch-spline-conv matminer -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ پکیج‌ها نصب شدند")

## 2. ایمپورت کتابخانه‌ها

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 3. مسیرهای داده (مطابق با `src/data/build_dataset.py` پروژه اصلی)

این مسیرها باید به دیتاست‌های آپلودشده در Kaggle اشاره کنند. ساختار فایل‌ها دقیقاً مطابق
ریپوی اصلی پروژه (`Metisa811/Predict-Phonon-Dispersion-by-GNN-Model`) است:
- **Force Constants**: فایل‌های `.fc` (فرمت Phonopy)
- **POSCAR**: فایل‌های `.psc`
- **Bands**: فایل‌های `.npy` (نوارهای فونون از قبل محاسبه‌شده)
- **Elastic constants**: یک JSON با C11, C12, ... برای هر ماده


In [ ]:
# ─── مسیرهای ورودی (با مسیر واقعی دیتاست‌های Kaggle خودتان جایگزین کنید) ───
FC_DIR       = '/kaggle/input/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR   = '/kaggle/input/pgcnndata/all_poscar/all_poscar'
BANDS_DIR    = '/kaggle/input/pgcnndata/extracted_bands_C'
ELASTIC_FILE = '/kaggle/input/features-dataset/mechanical_data_fixed.json'

# ─── کاندیدای Feature Set های اتمی (هر کدام یک CSV با ستون symbol + فیچرهای عددی) ───
FEATURE_SETS = {
    '0_RECOMMENDED_2025_BEST': '/kaggle/input/d/metisa81/feature-dataset-split/0_RECOMMENDED_2025_BEST.csv',
    '9_CLEAN_no_missing':      '/kaggle/input/d/metisa81/feature-dataset-split/9_CLEAN_no_missing.csv',
    '5_geometry_radius_volume':'/kaggle/input/d/metisa81/feature-dataset-split/5_geometry_radius_volume.csv',
    '2_phonon_PROVEN_best10':  '/kaggle/input/d/metisa81/feature-dataset-split/2_phonon_PROVEN_best10.csv',
}

for name, path in FEATURE_SETS.items():
    exists = '✅' if os.path.exists(path) else '❌ یافت نشد'
    print(f"{exists}  {name}  →  {path}")

## 4. پارسرهای داده (پورت‌شده از `src/data/parse_poscar.py` و `parse_force_constants.py`)

In [ ]:
def parse_poscar(path: str) -> dict:
    """پارس فایل POSCAR / .psc — دقیقا مطابق نسخه‌ی ریپوی اصلی پروژه."""
    with open(path, 'r') as f:
        lines = f.readlines()

    scaling = float(lines[1].strip())
    lattice = np.array(
        [list(map(float, lines[i].split())) for i in range(2, 5)]
    ) * scaling

    line5 = lines[5].strip().split()
    line6 = lines[6].strip().split()

    if line5[0].isalpha():
        elements   = line5
        counts     = list(map(int, line6))
        coord_line = 7
    else:
        counts     = list(map(int, line5))
        elements   = [f'X{i}' for i in range(len(counts))]
        coord_line = 6

    coord_type = lines[coord_line].strip()[0].upper()
    n_atoms    = sum(counts)

    positions = np.array([
        list(map(float, lines[i].split()[:3]))
        for i in range(coord_line + 1, coord_line + 1 + n_atoms)
    ])

    if coord_type == 'D':
        positions = positions @ lattice

    atom_elements = []
    for elem, cnt in zip(elements, counts):
        atom_elements.extend([elem] * cnt)

    return {
        'lattice':       lattice.astype(np.float32),
        'positions':     positions.astype(np.float32),
        'elements':      elements,
        'counts':        counts,
        'atom_elements': atom_elements,
        'n_atoms':       n_atoms,
        'volume':        float(np.abs(np.linalg.det(lattice))),
    }


def parse_force_constants(fc_path: str):
    """پارس فایل Phonopy FORCE_CONSTANTS — دقیقا مطابق نسخه‌ی ریپوی اصلی پروژه."""
    with open(fc_path, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]

    n_atoms = int(lines[0].split()[0])
    ifcs    = np.zeros((n_atoms, n_atoms, 3, 3), dtype=np.float64)

    idx = 1
    while idx < len(lines):
        parts = lines[idx].split()
        if len(parts) == 2:
            try:
                i, j = int(parts[0]) - 1, int(parts[1]) - 1
                idx += 1
                for row in range(3):
                    vals = list(map(float, lines[idx].split()))
                    ifcs[i, j, row, :] = vals[:3]
                    idx += 1
            except (ValueError, IndexError):
                idx += 1
        else:
            idx += 1

    return n_atoms, ifcs

print("✅ پارسرها آماده شدند")

## 5. تطبیق فایل‌ها و ساخت دیتاست خام (Raw Samples)

هر ماده باید همزمان فایل Force Constants، POSCAR و Bands داشته باشد.
این مرحله مستقل از Feature Set است — فقط یک‌بار اجرا می‌شود.


In [ ]:
fc_dict     = {Path(f).stem: f for f in Path(FC_DIR).glob('*.fc')}
poscar_dict = {Path(f).stem: f for f in Path(POSCAR_DIR).glob('*.psc')}
band_dict   = {Path(f).stem: np.load(f, allow_pickle=True) for f in Path(BANDS_DIR).glob('*.npy')}

common = sorted(set(fc_dict) & set(poscar_dict) & set(band_dict))
print(f"تعداد مواد مشترک (با هر سه فایل): {len(common)}")

shapes = [np.array(band_dict[f]).shape for f in common]
target_shape = Counter(shapes).most_common(1)[0][0]
print(f"شکل استاندارد بردار فونون: {target_shape}")

with open(ELASTIC_FILE) as f:
    elastic_json = json.load(f)
ELASTIC_KEYS = ['C11','C12','C13','C33','C44','C66',
                'B_Hill','G_Hill','E_Hill','Poisson_Hill',
                'Pugh_ratio','Debye_temperature']
elastic_data = {}
for entry in elastic_json:
    mat = entry.get('material', '')
    vals = [float(entry.get(k, 0) or 0) for k in ELASTIC_KEYS]
    elastic_data[mat] = np.array(vals, dtype=np.float32)

raw_samples = []
failed = []
for formula in tqdm(common, desc="ساخت دیتاست خام"):
    try:
        band_arr = np.array(band_dict[formula])
        if band_arr.shape != target_shape:
            continue
        poscar = parse_poscar(str(poscar_dict[formula]))
        n_atoms, ifcs = parse_force_constants(str(fc_dict[formula]))
        if n_atoms != poscar['n_atoms']:
            continue

        raw_samples.append({
            'formula':           formula,
            'n_atoms':           n_atoms,
            'lattice':           poscar['lattice'],
            'positions':         poscar['positions'],
            'atom_elements':     poscar['atom_elements'],
            'force_constants':   ifcs.astype(np.float32),
            'elastic_constants': elastic_data.get(formula, np.zeros(len(ELASTIC_KEYS), np.float32)),
            'has_elastic':       formula in elastic_data,
            'y_phonon':          band_arr.astype(np.float32),
        })
    except Exception as e:
        failed.append((formula, str(e)))

print(f"\n✅ موفق: {len(raw_samples)}  |  ❌ ناموفق: {len(failed)}")
if failed[:5]:
    print("نمونه خطاها:", failed[:5])

## 6. تقسیم Train / Val / Test (ثابت برای همه‌ی Feature Set ها)

برای مقایسه‌ی عادلانه، split باید بین همه‌ی آزمایش‌ها یکسان باشد (همان seed).


In [ ]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.permutation(len(raw_samples))

n_total = len(raw_samples)
n_tr = int(0.8 * n_total)
n_va = int(0.1 * n_total)

train_idx = idx[:n_tr]
val_idx   = idx[n_tr:n_tr+n_va]
test_idx  = idx[n_tr+n_va:]

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

## 7. معماری مدل: Dual Graph GNN (بهترین مدل تا الان — MAE=0.48 روی Matbench)

این دقیقاً همان معماری استفاده‌شده در `bestbest ce9993` است که بهترین نتیجه را در آزمایش‌های قبلی داشت:

- **Bond Graph**: nodes = پیوندهای شیمیایی (تا 8Å)، edges = زاویه بین پیوندها
- **Atom Graph**: nodes = اتم‌ها (سوپرسل 3×3×3)، edges = فاصله بین اتم‌ها
- **CustomMessagePassing**: لایه‌ی Attention مشابه GATv2 (سبک‌تر)
- **Pooling**: ترکیب Set2Set + Mean + Max برای هر دو گراف


In [ ]:
CUTOFF = 8.0

def structure_to_bond_graph(positions, target_value):
    """Bond Graph: nodes=bonds, edges=angles."""
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        y = torch.tensor(target_value, dtype=torch.float).flatten()
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)

    y = torch.tensor(target_value, dtype=torch.float).flatten()
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)


def structure_to_atom_graph(atom_elements, positions, target_value, df_features, n_atom_features):
    """Atom Graph: nodes=atoms, edges=distances. ساده‌سازی شده (بدون سوپرسل برای سرعت روی دیتاست واقعی)."""
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(n_atom_features, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    y = torch.tensor(target_value, dtype=torch.float).flatten()
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)

print("✅ توابع ساخت گراف آماده شدند")

In [ ]:
class CustomMessagePassing(nn.Module):
    """لایه‌ی Message Passing با Attention (مشابه GATv2 ولی سبک‌تر)."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphGNN(nn.Module):
    """مدل Dual Graph — معماری برنده در آزمایش‌های قبلی (bestbest ce9993)."""
    def __init__(self, n_bond_features=6, n_atom_features=13, edge_dim=1, output_dim=1):
        super().__init__()

        if n_atom_features <= 10:
            hidden = 128
        elif n_atom_features <= 20:
            hidden = 96
        elif n_atom_features <= 50:
            hidden = 64
        else:
            hidden = 50

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(5)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(5)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(2)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(2)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=1)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool

        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        final_input = 8 * hidden + hidden // 4
        self.final_mlp = nn.Sequential(
            nn.Linear(final_input, 256), nn.LayerNorm(256), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(64, output_dim)
        )

    def forward(self, bond_data, atom_data):
        x_bond = self.bond_embedding(bond_data.x)
        edge_features_bond = self.bond_edge_embedding(bond_data.edge_attr)
        for i, (msg, ln) in enumerate(zip(self.bond_message_layers, self.bond_layer_norms)):
            x_res = x_bond
            x_bond = msg(x_bond, bond_data.edge_index, edge_features_bond)
            x_bond = ln(x_bond)
            if i > 0:
                x_bond = x_bond + self.bond_residual_weight * x_res
            x_bond = F.silu(x_bond)

        attn_bond = self.bond_attention(x_bond)
        x_bond_w = x_bond * attn_bond
        bond_set2set = self.set2set_pool(x_bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        x_atom = self.atom_embedding(atom_data.x)
        edge_features_atom = self.atom_edge_embedding(atom_data.edge_attr)
        for i, (msg, ln) in enumerate(zip(self.atom_message_layers, self.atom_layer_norms)):
            x_res = x_atom
            x_atom = msg(x_atom, atom_data.edge_index, edge_features_atom)
            x_atom = ln(x_atom)
            if i > 0:
                x_atom = x_atom + self.atom_residual_weight * x_res
            x_atom = F.silu(x_atom)

        attn_atom = self.atom_attention(x_atom)
        x_atom_w = x_atom * attn_atom
        atom_set2set = self.set2set_pool(x_atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        combined = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            global_features
        ], dim=1)

        return self.final_mlp(combined)

print("✅ معماری DualGraphGNN تعریف شد")

## 8. توابع آموزش و ارزیابی

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for bond_data, atom_data in loader:
        bond_data, atom_data = bond_data.to(device), atom_data.to(device)
        optimizer.zero_grad()
        pred = model(bond_data, atom_data)
        loss = F.mse_loss(pred, bond_data.y.view(pred.shape))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * bond_data.num_graphs
    return total_loss / len(loader.dataset)


def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bond_data, atom_data in loader:
            bond_data, atom_data = bond_data.to(device), atom_data.to(device)
            pred = model(bond_data, atom_data)
            preds.append(pred.cpu().numpy())
            targets.append(bond_data.y.view(pred.shape).cpu().numpy())
    preds = np.concatenate(preds, axis=0)
    targets = np.concatenate(targets, axis=0)
    return mean_absolute_error(targets.flatten(), preds.flatten())

print("✅ توابع آموزش/ارزیابی آماده شدند")

## 9. حلقه‌ی مقایسه‌ی Feature Set ها

برای هر Feature Set:
1. فیچرهای اتمی لود می‌شوند
2. گراف‌های Bond و Atom برای کل دیتاست ساخته می‌شوند (یک‌بار، قبل از training)
3. مدل Dual Graph GNN از صفر آموزش می‌بیند
4. MAE روی Validation و Test ثبت می‌شود

⏱️ این مرحله کند است (ساخت گراف + آموزش برای هر Feature Set) — اما برای مقایسه‌ی دقیق لازم است.


In [ ]:
EPOCHS = 300          # برای مقایسه سریع‌تر؛ برای مدل نهایی می‌توان به 1000 افزایش داد
BATCH_SIZE = 16        # دیتاست کوچک‌تر از Matbench است (358 ماده) پس batch کوچک‌تر
PATIENCE = 50

results = {}

for feature_name, feature_path in FEATURE_SETS.items():
    print(f"\n{'='*70}")
    print(f"🔬 Feature Set: {feature_name}")
    print(f"{'='*70}")

    if not os.path.exists(feature_path):
        print(f"⚠️ فایل پیدا نشد، رد شد: {feature_path}")
        continue

    # --- لود فیچرها ---
    df_raw = pd.read_csv(feature_path)
    symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
    feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
    feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]

    df_features = df_raw[feature_cols].copy()
    df_features.index = df_raw[symbol_col]
    n_atom_features = len(feature_cols)
    print(f"   فیچرها: {n_atom_features} → {feature_cols[:5]}{'...' if n_atom_features > 5 else ''}")

    # --- ساخت گراف‌ها برای کل دیتاست ---
    bond_graphs, atom_graphs = [], []
    for s in tqdm(raw_samples, desc="ساخت گراف‌ها", leave=False):
        target = s['y_phonon']  # می‌تواند کامل یا یک سکالار خلاصه باشد بسته به فرمت بانک داده
        bond_graphs.append(structure_to_bond_graph(s['positions'], target))
        atom_graphs.append(structure_to_atom_graph(
            s['atom_elements'], s['positions'], target, df_features, n_atom_features))

    output_dim = bond_graphs[0].y.numel()

    train_set = [(bond_graphs[i], atom_graphs[i]) for i in train_idx]
    val_set   = [(bond_graphs[i], atom_graphs[i]) for i in val_idx]
    test_set  = [(bond_graphs[i], atom_graphs[i]) for i in test_idx]

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

    # --- مدل و آموزش ---
    model = DualGraphGNN(n_bond_features=6, n_atom_features=n_atom_features,
                          edge_dim=1, output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=0.01, epochs=EPOCHS, steps_per_epoch=max(1, len(train_loader)),
        pct_start=0.3, anneal_strategy='cos', div_factor=25.0, final_div_factor=1e5)

    best_val_mae = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        if epoch % 5 == 0 or epoch == EPOCHS - 1:
            val_mae = evaluate(model, val_loader, device)
            if val_mae < best_val_mae:
                best_val_mae = val_mae
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 5
            if epoch % 20 == 0:
                print(f"   Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val MAE: {val_mae:.4f}")
            if patience_counter >= PATIENCE:
                print(f"   ⚠️ Early stop at epoch {epoch}")
                break

    # --- ارزیابی نهایی روی Test با بهترین وزن‌ها ---
    if best_state is not None:
        model.load_state_dict(best_state)
    test_mae = evaluate(model, test_loader, device)

    results[feature_name] = {
        'n_features': n_atom_features,
        'best_val_mae': best_val_mae,
        'test_mae': test_mae,
    }
    print(f"   ✅ {feature_name} → Val MAE: {best_val_mae:.4f} | Test MAE: {test_mae:.4f}")

## 10. جدول مقایسه‌ی نهایی و انتخاب بهترین Feature Set

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('test_mae')
results_df.index.name = 'feature_set'
print(results_df.to_string())

best_feature_set = results_df.index[0]
print(f"\n🏆 بهترین Feature Set: {best_feature_set}")
print(f"   Test MAE: {results_df.loc[best_feature_set, 'test_mae']:.4f}")
print(f"   تعداد فیچر: {int(results_df.loc[best_feature_set, 'n_features'])}")

results_df.to_csv('/kaggle/working/feature_comparison_results.csv')
print("\n💾 نتایج ذخیره شد: feature_comparison_results.csv")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].bar(results_df.index, results_df['test_mae'], color='steelblue')
ax[0].set_ylabel('Test MAE')
ax[0].set_title('مقایسه‌ی MAE برای هر Feature Set')
ax[0].tick_params(axis='x', rotation=30)

ax[1].scatter(results_df['n_features'], results_df['test_mae'], s=100, color='darkorange')
for name, row in results_df.iterrows():
    ax[1].annotate(name, (row['n_features'], row['test_mae']), fontsize=8, xytext=(5,5), textcoords='offset points')
ax[1].set_xlabel('تعداد فیچر')
ax[1].set_ylabel('Test MAE')
ax[1].set_title('تعداد فیچر در مقابل دقت')

plt.tight_layout()
plt.savefig('/kaggle/working/feature_comparison_plot.png', dpi=150)
plt.show()

## 11. آموزش نهایی مدل با بهترین Feature Set (Full Training)

با Feature Set برتر، مدل را با تعداد epoch بیشتر (1000، مطابق آزمایش‌های قبلی) و patience بالاتر
دوباره آموزش می‌دهیم تا به بهترین دقت ممکن برسیم.


In [ ]:
print(f"🚀 آموزش نهایی با Feature Set برتر: {best_feature_set}")

feature_path = FEATURE_SETS[best_feature_set]
df_raw = pd.read_csv(feature_path)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features_final = df_raw[feature_cols].copy()
df_features_final.index = df_raw[symbol_col]
n_atom_features_final = len(feature_cols)

bond_graphs_final, atom_graphs_final = [], []
for s in tqdm(raw_samples, desc="ساخت گراف‌های نهایی"):
    target = s['y_phonon']
    bond_graphs_final.append(structure_to_bond_graph(s['positions'], target))
    atom_graphs_final.append(structure_to_atom_graph(
        s['atom_elements'], s['positions'], target, df_features_final, n_atom_features_final))

output_dim_final = bond_graphs_final[0].y.numel()

train_set_f = [(bond_graphs_final[i], atom_graphs_final[i]) for i in train_idx]
val_set_f   = [(bond_graphs_final[i], atom_graphs_final[i]) for i in val_idx]
test_set_f  = [(bond_graphs_final[i], atom_graphs_final[i]) for i in test_idx]

train_loader_f = DataLoader(train_set_f, batch_size=BATCH_SIZE, shuffle=True)
val_loader_f   = DataLoader(val_set_f,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_f  = DataLoader(test_set_f,  batch_size=BATCH_SIZE, shuffle=False)

FINAL_EPOCHS = 1000
FINAL_PATIENCE = 150

final_model = DualGraphGNN(n_bond_features=6, n_atom_features=n_atom_features_final,
                            edge_dim=1, output_dim=output_dim_final).to(device)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.01, epochs=FINAL_EPOCHS, steps_per_epoch=max(1, len(train_loader_f)),
    pct_start=0.3, anneal_strategy='cos', div_factor=25.0, final_div_factor=1e5)

best_val_mae_final = float('inf')
best_state_final = None
patience_counter = 0

for epoch in range(FINAL_EPOCHS):
    train_loss = train_epoch(final_model, train_loader_f, optimizer, scheduler, device)
    if epoch % 5 == 0 or epoch == FINAL_EPOCHS - 1:
        val_mae = evaluate(final_model, val_loader_f, device)
        if val_mae < best_val_mae_final:
            best_val_mae_final = val_mae
            best_state_final = {k: v.clone() for k, v in final_model.state_dict().items()}
            patience_counter = 0
            torch.save(best_state_final, '/kaggle/working/best_final_model.pt')
        else:
            patience_counter += 5
        if epoch % 20 == 0:
            print(f"Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Val MAE: {val_mae:.4f} ⭐ Best: {best_val_mae_final:.4f}")
        if patience_counter >= FINAL_PATIENCE:
            print(f"⚠️ Early stop at epoch {epoch}")
            break

final_model.load_state_dict(best_state_final)
final_test_mae = evaluate(final_model, test_loader_f, device)

print(f"\n{'='*60}")
print(f"🏁 نتیجه نهایی")
print(f"{'='*60}")
print(f"Feature Set:  {best_feature_set}")
print(f"تعداد فیچر:    {n_atom_features_final}")
print(f"Val MAE:       {best_val_mae_final:.4f}")
print(f"Test MAE:      {final_test_mae:.4f}")
print(f"\n💾 مدل نهایی ذخیره شد: /kaggle/working/best_final_model.pt")

## 12. جمع‌بندی

| مرحله | نتیجه |
|------|-------|
| Feature Sets آزمایش‌شده | (نتایج بالا را ببینید) |
| بهترین Feature Set | `best_feature_set` (متغیر بالا) |
| معماری استفاده‌شده | Dual Graph GNN (Bond + Atom Graph با Attention) |
| Test MAE نهایی | `final_test_mae` (متغیر بالا) |

### مراحل بعدی
1. این نتایج را در فایل `08 - نقشه راه مقاله.md` و `10 - روند Notebook های Kaggle.md` در Obsidian ثبت کنید.
2. Feature Set برتر را در `src/data/build_dataset.py` ریپوی اصلی به‌عنوان `csv_file` پیش‌فرض قرار دهید.
3. برای رسیدن به هدف MAE < 0.10، باید Physics-Informed Loss (طبق README ریپو) به این مدل اضافه شود — این مرحله بعدی پروژه است.
